# 4 · Preparación del dataset para el modelado

**Proyecto:** Predicción de readmisión hospitalaria en pacientes diabéticos
**Dataset:** *Diabetes 130-US hospitals for years 1999-2008* (UCI Machine Learning Repository)
**Autor:** Diego Rodríguez Díaz del Campo

---

### Objetivo de este notebook

Convertir el dataset limpio en un conjunto de datos **listo para entrenar un modelo**,
aplicando las decisiones que el EDA dejó preparadas (apartados 5.6, 6.3 y 7 del
notebook 03). Cada transformación se justifica con la evidencia del EDA; nada se hace
por rutina.

El modelado **no** forma parte de este proyecto: este notebook termina con los
conjuntos `train` y `test` guardados en disco, que serán la entrada de un proyecto
futuro.

| | |
|---|---|
| **Entrada** | `data/processed/diabetic_data_clean.csv` (101.766 × 35) y `data/raw/diabetic_data.csv` (solo para recuperar `patient_nbr`) |
| **Variable objetivo** | `readmitted` |
| **Salida** | `data/processed/train.csv` y `data/processed/test.csv` |

### Contenido

1. Configuración, carga y recuperación de `patient_nbr`
2. Exclusión de pacientes fallecidos y en cuidados paliativos
3. Agrupación de categorías poco frecuentes y de alta cardinalidad
4. Eliminación de variables sin señal
5. Formulación de la variable objetivo
6. Codificación de las variables categóricas
7. División en `train` y `test` sin fuga de datos entre pacientes
8. Guardado y conclusiones

## 1 · Configuración, carga y recuperación de `patient_nbr`

### 1.1 Carga y recuperación de tipos

Se parte del dataset limpio (notebook 02). Como se explicó en el notebook 03, el CSV
**no conserva los `dtype`**: los tres códigos administrativos (`admission_type_id`,
`discharge_disposition_id`, `admission_source_id`) y los códigos de diagnóstico vuelven
a leerse como números y hay que declararlos de nuevo como categorías. Se repite aquí la
misma corrección que en el notebook 03, sin volver a discutirla.

Solo se importa lo que este apartado necesita (`pandas`, `numpy`, `pathlib`). Las
librerías gráficas y `scikit-learn` se importarán en el apartado en que hagan falta,
para que quede claro *para qué* se usa cada una.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'D:\Cosas Diego\Universidad\Proyectos personales\proyecto 1')
RAW = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'

df = pd.read_csv(PROC / 'diabetic_data_clean.csv', low_memory=False)

print(f'Dimensiones: {df.shape}')
df.head()

Dimensiones: (101766, 35)


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,41,...,No,No,No,No,No,No,No,No,No,NO
1,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,59,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,11,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,44,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,51,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


In [2]:
# Códigos administrativos: son etiquetas, no cantidades
columnas_categoricas_numericas = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

df[columnas_categoricas_numericas] = df[columnas_categoricas_numericas].astype('category')

# Códigos de diagnóstico: el código 700 no es "mayor" que el 200
df[['diag_1', 'diag_2', 'diag_3']] = df[['diag_1', 'diag_2', 'diag_3']].astype('category')

df.dtypes

race                          object
gender                        object
age                           object
admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
time_in_hospital               int64
payer_code                    object
medical_specialty             object
num_lab_procedures             int64
num_procedures                 int64
num_medications                int64
number_outpatient              int64
number_emergency               int64
number_inpatient               int64
diag_1                      category
diag_2                      category
diag_3                      category
number_diagnoses               int64
max_glu_serum                 object
A1Cresult                     object
metformin                     object
repaglinide                   object
nateglinide                   object
glimepiride                   object
glipizide                     object
glyburide                     object
p

### 1.2 Por qué hay que recuperar `patient_nbr`

En la limpieza se eliminó `patient_nbr` porque un identificador de paciente no es una
variable predictora. Sin embargo, el notebook 01 mostró que los 101.766 ingresos
corresponden a **71.518 pacientes distintos**: hay pacientes con 2, 3 y hasta 40
ingresos en el dataset.

Esto tiene una consecuencia directa sobre el apartado 7. Si se dividen las *filas* al
azar entre `train` y `test`, los ingresos de un mismo paciente quedarán repartidos en
ambos conjuntos. El modelo vería en `test` a pacientes que ya conoce de `train`, con sus
mismas características demográficas y clínicas, y su rendimiento parecería mejor de lo
que es en realidad. Es una forma de **fuga de datos** (*data leakage*): la división debe
hacerse **por paciente**, y para eso hace falta el identificador.

### Cómo se recupera: por posición, no por clave

El dataset limpio ya no tiene ninguna columna que sirva de clave para cruzar con el
original (`encounter_id` también se eliminó). Pero la limpieza **eliminó 15 columnas y
0 filas, y nunca reordenó**: la fila *i* del dataset limpio es la fila *i* del original.
Por tanto, `patient_nbr` se puede copiar del original **por posición**.

Es una hipótesis razonable, pero es una hipótesis. Antes de pegar la columna se comprueba
con dos pruebas:

1. Que ambos ficheros tienen exactamente **101.766 filas**.
2. Que una columna que sobrevivió intacta a la limpieza, `time_in_hospital`, **coincide
   fila a fila** en los dos ficheros.

Del original solo se leen las dos columnas necesarias (`usecols`): `patient_nbr`, que es
lo que se quiere recuperar, y `time_in_hospital`, que sirve de columna de control.

In [3]:
original = pd.read_csv(
    RAW / 'diabetic_data.csv',
    usecols=['patient_nbr', 'time_in_hospital']
)

print(f'Filas en el limpio:   {len(df)}')
print(f'Filas en el original: {len(original)}')

Filas en el limpio:   101766
Filas en el original: 101766


Las dos tablas tienen el mismo número de filas. Ahora la comprobación decisiva: que
`time_in_hospital` coincide **posición a posición**.

Se compara con `.to_numpy()` en vez de comparar las dos Series directamente. La razón es
que pandas, al operar entre dos Series, las **alinea por índice**; aquí interesa
justamente lo contrario, comparar por posición pura, porque es lo que se va a asumir al
pegar la columna. Con los arrays de NumPy la comparación es estrictamente posicional.

In [4]:
coinciden = (df['time_in_hospital'].to_numpy() == original['time_in_hospital'].to_numpy())

print(f'Filas que coinciden: {coinciden.sum()} de {len(coinciden)}')
print(f'¿Coinciden todas?    {coinciden.all()}')

Filas que coinciden: 101766 de 101766
¿Coinciden todas?    True


Las dos comprobaciones son favorables, así que se pega `patient_nbr` por posición. Se
inserta como **primera columna** con `insert(0, ...)`, porque es un identificador y no una
variable más: así queda claro a simple vista que no forma parte de las predictoras.

Como verificación final, el número de pacientes distintos debe ser **71.518**, la cifra
obtenida en el notebook 01 sobre el dataset original. Si diera otra cosa, la columna se
habría pegado desalineada.

In [5]:
df.insert(0, 'patient_nbr', original['patient_nbr'].to_numpy())

print(f'Dimensiones:        {df.shape}')
print(f'Pacientes distintos: {df["patient_nbr"].nunique()}')
df.head()

Dimensiones:        (101766, 36)
Pacientes distintos: 71518


,patient_nbr,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,glipizide,glyburide,pioglitazone,rosiglitazone,acarbose,insulin,glyburide-metformin,change,diabetesMed,readmitted
0,8222157,Caucasian,Female,[0-10),6,25,1,1,Unknown,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,NO
1,55629189,Caucasian,Female,[10-20),1,1,7,3,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,>30
2,86047875,AfricanAmerican,Female,[20-30),1,1,7,2,Unknown,Unknown,...,Steady,No,No,No,No,No,No,No,Yes,NO
3,82442376,Caucasian,Male,[30-40),1,1,7,2,Unknown,Unknown,...,No,No,No,No,No,Up,No,Ch,Yes,NO
4,42519267,Caucasian,Male,[40-50),1,1,7,1,Unknown,Unknown,...,Steady,No,No,No,No,Steady,No,Ch,Yes,NO


### Interpretación del apartado 1

Las tres verificaciones han sido favorables:

| Comprobación | Resultado esperado | Obtenido |
|---|---|---|
| Filas en ambos ficheros | 101.766 | 101.766 y 101.766 |
| `time_in_hospital` coincide fila a fila | todas | 101.766 de 101.766 (`True`) |
| Pacientes distintos tras pegar `patient_nbr` | 71.518 (notebook 01) | 71.518 |

La hipótesis de que la limpieza no alteró el orden de las filas queda **confirmada con
evidencia**, no asumida. El dataset de trabajo tiene ahora **101.766 × 36**: las 35
columnas del limpio más `patient_nbr` en primera posición.

Dos advertencias para el resto del notebook:

- `patient_nbr` es un **identificador**, no una variable predictora. Se conserva
  únicamente para hacer la división por paciente del apartado 7 y **no** debe entrar
  en el modelo: un número de historia clínica no dice nada del riesgo de readmisión.
- La cifra de 71.518 pacientes es válida **ahora**; cuando en el apartado 2 se excluyan
  los fallecidos y los pacientes en cuidados paliativos, cambiará y habrá que
  recalcularla.

## 2 · Exclusión de pacientes fallecidos y en cuidados paliativos

### El problema: un `NO` que no significa "no readmitido"

La variable objetivo `readmitted` toma el valor `NO` cuando el paciente no volvió a
ingresar. Pero hay un grupo de pacientes para los que ese `NO` **no es una decisión
clínica ni un buen resultado, sino una imposibilidad física**: los que fallecieron
durante el ingreso. Un paciente que muere en el hospital no puede ser readmitido.

Si estas filas se quedan en el dataset, el modelo aprendería que *"morir en el hospital
protege de la readmisión"*: un patrón numéricamente cierto y clínicamente absurdo. Y el
EDA ya mostró el efecto: el tramo `[90-100)` de `age` tenía un `NO` anormalmente alto
que se explicaba en parte por los fallecidos (apartado 5.5 del notebook 03).

### Qué códigos se excluyen y por qué

Según `IDS_mapping.csv`, `discharge_disposition_id` tiene cuatro códigos de fallecimiento
y dos de cuidados paliativos (*hospice*). Los recuentos son los del dataset limpio:

| Código | Descripción | Ingresos | `<30` / `>30` / `NO` |
|---|---|---|---|
| 11 | Expired | 1.642 | 0 / 0 / 1.642 |
| 19 | Expired at home. Medicaid only, hospice | 8 | 0 / 0 / 8 |
| 20 | Expired in a medical facility. Medicaid only, hospice | 2 | 0 / 0 / 2 |
| 21 | Expired, place unknown. Medicaid only, hospice | 0 | — |
| 13 | Hospice / home | 399 | 19 / 36 / 344 |
| 14 | Hospice / medical facility | 372 | 24 / 7 / 341 |
| | **Total** | **2.423** | |

Los tres códigos de fallecimiento presentes (11, 19, 20) son `NO` en el **100 %** de los
casos: la evidencia es inequívoca. El código 21 no aparece en el dataset, pero se incluye
en la lista para que la regla sea completa y no dependa de qué códigos hay hoy.

Los códigos de *hospice* (13, 14) son un caso distinto y merecen justificación propia.
El alta a cuidados paliativos implica un pronóstico de vida limitado y, sobre todo, un
**cambio de objetivo asistencial**: se renuncia a la intervención curativa, de modo que un
empeoramiento no conduce a un reingreso hospitalario sino a manejo del confort. No son
todos `NO` (86 % y 92 %), pero su `NO` mayoritario obedece a la misma lógica que el de los
fallecidos: **no informa de la calidad del alta ni del riesgo del paciente**, que es
exactamente lo que el modelo debe aprender. Por eso se excluyen junto con ellos.

### Se excluyen ingresos, no pacientes

Un paciente que falleció en su último ingreso puede tener ingresos anteriores en el
dataset. Esos ingresos anteriores son **válidos**: en ellos el paciente fue dado de alta
vivo y su `readmitted` refleja lo que realmente ocurrió después. Por tanto se eliminan
solo las **filas** con esos códigos de alta, no todas las filas de esos `patient_nbr`.

### Qué se comprueba

1. Que la máscara selecciona exactamente **2.423** filas (la cifra del contraste
   `[90-100)` del notebook 03).
2. Que tras filtrar quedan **99.343** filas y se recalcula el número de pacientes
   distintos, que ya no será 71.518.
3. Que los seis códigos han desaparecido de `discharge_disposition_id`.

Sobre el índice: al eliminar filas, el índice de `df` queda con huecos. Se reinicia con
`reset_index(drop=True)` porque a partir de aquí ya no hace falta mantener la
correspondencia posicional con el fichero original (`patient_nbr` ya está pegado) y un
índice continuo evita sorpresas en operaciones posteriores.

In [6]:
codigos_excluir = [11,13,14,19,20,21]

mascara_excluir = df['discharge_disposition_id'].isin(codigos_excluir)
int(mascara_excluir.sum())

2423

In [7]:
filas_marcadas = df[mascara_excluir]

pd.crosstab(filas_marcadas['discharge_disposition_id'], filas_marcadas['readmitted'])

readmitted,<30,>30,NO
discharge_disposition_id,,,
11,0,0,1642
13,19,36,344
14,24,7,341
19,0,0,8
20,0,0,2


In [8]:
# Eliminamos los ingresos de fallecidos y hospice y reiniciamos el índice
df = df[~mascara_excluir].reset_index(drop=True)
df.shape

(99343, 36)

In [9]:
bool(df['discharge_disposition_id'].isin(codigos_excluir).any())

False

In [10]:
df['patient_nbr'].nunique()

69990

### Interpretación del apartado 2

Las tres comprobaciones coinciden con lo previsto:

| Comprobación | Esperado | Obtenido |
|---|---|---|
| Filas que marca la máscara | 2.423 | 2.423 |
| Forma de `df` tras el filtrado | (99.343, 36) | (99.343, 36) |
| Algún código de exclusión sigue presente | `False` | `False` |

La tabla cruzada confirma la evidencia con recuentos, no con porcentajes redondeados: los
tres códigos de fallecimiento presentes (11, 19, 20) son `NO` en **1.652 de 1.652** ingresos,
sin una sola excepción. Un `1,00` redondeado habría sido compatible con unos pocos
readmitidos ocultos; el `0 · 0 · 1642` no deja margen.

### Pacientes distintos: de 71.518 a 69.990

El número de pacientes baja en **1.528**. Como se excluyeron 2.423 ingresos, el reparto es:

- **1.538 ingresos** pertenecían a pacientes cuyo *único* registro en el dataset era el
  ingreso en que fallecieron o pasaron a cuidados paliativos (10 de esos pacientes tenían
  dos ingresos con código de exclusión, por ejemplo un alta a *hospice* seguida del
  fallecimiento). Esos pacientes desaparecen del dataset.
- **885 ingresos** pertenecían a pacientes que conservan ingresos anteriores, dados de alta
  vivos. Esos pacientes siguen en el dataset con sus registros válidos, que es exactamente
  lo que pretendía la decisión de excluir *ingresos* y no *pacientes*.

La cifra de referencia para el resto del notebook, y en particular para la división por
paciente del apartado 7, pasa a ser **69.990 pacientes en 99.343 ingresos**.

### Un residuo técnico: las categorías vacías

Eliminar filas no elimina categorías. `discharge_disposition_id` es de tipo `category`, y
su lista de categorías declaradas sigue teniendo las **26** originales aunque solo **21**
tienen filas: los códigos 11, 13, 14, 19 y 20 siguen existiendo como categorías con cero
ingresos. `pd.crosstab` las ha ignorado, pero otras operaciones no lo hacen (`groupby`
con `observed=False`, `value_counts`, o cualquier codificación *one-hot* generaría columnas
de ceros). Se resuelve al principio del apartado 3, que es donde se revisa la lista de
categorías de cada variable.


## 3 · Agrupación de categorías poco frecuentes y de alta cardinalidad

### Por qué agrupar

El EDA dejó dos problemas distintos con las variables categóricas, y ambos tienen la misma
consecuencia para un modelo: categorías con tan pocos ingresos que **no es posible aprender
nada fiable de ellas**, y que además multiplican el número de columnas en la codificación.

1. **Categorías raras en variables de cardinalidad baja o media.** `admission_type_id`
   tenía en el EDA códigos con 10 y 21 ingresos; `discharge_disposition_id`, `admission_source_id` y
   `payer_code` tienen varias categorías por debajo de 100. Un porcentaje de readmisión
   calculado sobre 10 ingresos es ruido (el EDA lo vio con `acarbose`: 13 ingresos
   ajustados daban un 23 % sin significado). La decisión del apartado 5.6 del notebook 03
   fue agruparlas en una categoría **`Otros`** con un umbral de **100 ingresos**.

2. **Alta cardinalidad.** `diag_1`, `diag_2` y `diag_3` tienen 716, 748 y 789 códigos
   CIE-9 distintos, y `medical_specialty` 73 especialidades. Codificarlas tal cual con
   *one-hot* añadiría más de 2.300 columnas, casi todas con un puñado de unos. Para los
   diagnósticos la alternativa estándar en la literatura clínica (y la que usaron Strack
   et al., 2014, con este mismo dataset) es agrupar los códigos en **capítulos CIE-9**
   (circulatorio, respiratorio, digestivo, diabetes, lesiones, ...), que reducen ~750
   valores a menos de 20 conservando el significado clínico.

El apartado sigue este orden, de lo sencillo a lo complejo:

- **3.1** Eliminar las categorías vacías que dejó el apartado 2.
- **3.2** `Otros` en `admission_type_id`, `discharge_disposition_id`, `admission_source_id`
  y `payer_code`.
- **3.3** Capítulos CIE-9 para `diag_1`, `diag_2` y `diag_3`.
- **3.4** `medical_specialty`.

### 3.1 · Categorías vacías

Antes de agrupar nada hay que dejar las listas de categorías en un estado coherente: si
`discharge_disposition_id` sigue declarando los códigos 11, 13, 14, 19 y 20 con cero
ingresos, cualquier recuento por categoría los mostrará como filas a 0 y la regla de
`Otros` (menos de 100 ingresos) los atraparía como si fueran categorías reales.

pandas ofrece el método `.cat.remove_unused_categories()`, que devuelve la misma Serie con
la lista de categorías reducida a las que tienen al menos un valor. No cambia ningún dato:
solo la lista declarada.

**Qué se comprueba:** que el número de categorías declaradas pasa de **26 a 21** y que
coincide con el número de valores distintos presentes (`nunique()`). Es la única variable
afectada, porque es la única en la que se han eliminado filas por su valor; en el resto,
las categorías se construyeron a partir de los datos al cargar y todas tienen ingresos.
